## Weather ETL Pipeline
Extracting live weather data from OpenWeather API, cleaning it with pandas, and storing it for analysis.

In [ ]:
import requests
import pandas as pd
from datetime import datetime

In [ ]:
API_KEY = "40abb55ac9d9fe527637103924519606"

In [ ]:
city = "Lagos"
url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"

response = requests.get(url)
print(response.status_code)

200


In [ ]:
data = response.json()
print(data)

{'coord': {'lon': 3.75, 'lat': 6.5833}, 'weather': [{'id': 500, 'main': 'Rain', 'description': 'light rain', 'icon': '10d'}], 'base': 'stations', 'main': {'temp': 26.52, 'feels_like': 26.52, 'temp_min': 26.52, 'temp_max': 26.52, 'pressure': 1015, 'humidity': 82, 'sea_level': 1015, 'grnd_level': 1015}, 'visibility': 10000, 'wind': {'speed': 2.2, 'deg': 244, 'gust': 3.49}, 'rain': {'1h': 0.44}, 'clouds': {'all': 100}, 'dt': 1783934384, 'sys': {'country': 'NG', 'sunrise': 1783920978, 'sunset': 1783965883}, 'timezone': 3600, 'id': 2332453, 'name': 'Lagos', 'cod': 200}


In [ ]:
city_name = data["name"]
temperature = data["main"]["temp"]
humidity = data["main"]["humidity"]
weather_condition = data["weather"][0]["main"]
wind_speed = data["wind"]["speed"]
date_time = data["dt"]

print(city_name, temperature, humidity, weather_condition, wind_speed, date_time)

Lagos 26.52 82 Rain 2.2 1783934384


In [ ]:
readable_time = datetime.fromtimestamp(date_time)
print(readable_time)

2026-07-13 09:19:44


## Task 1: Extract
Pulling live weather data for 3 Nigerian cities from the OpenWeather API.

In [ ]:
cities = ["Lagos", "Abuja", "Ibadan"]

In [ ]:
weather_data = []

for city in cities:
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"
    response = requests.get(url)
    data = response.json()
    weather_data.append(data)

print(len(weather_data))

3


## Task 2: Transform
Cleaning the raw JSON responses into a structured dataset with pandas.

In [ ]:
cleaned_data = []

for entry in weather_data:
    record = {
        "city": entry["name"],
        "temperature_c": entry["main"]["temp"],
        "humidity_pct": entry["main"]["humidity"],
        "weather_condition": entry["weather"][0]["main"],
        "wind_speed_mps": entry["wind"]["speed"],
        "date_time": datetime.fromtimestamp(entry["dt"])
    }
    cleaned_data.append(record)

print(cleaned_data)

[{'city': 'Lagos', 'temperature_c': 26.52, 'humidity_pct': 82, 'weather_condition': 'Rain', 'wind_speed_mps': 2.2, 'date_time': datetime.datetime(2026, 7, 13, 9, 28, 8)}, {'city': 'Abuja', 'temperature_c': 22.73, 'humidity_pct': 91, 'weather_condition': 'Clouds', 'wind_speed_mps': 2.53, 'date_time': datetime.datetime(2026, 7, 13, 9, 28, 34)}, {'city': 'Ibadan', 'temperature_c': 26.29, 'humidity_pct': 83, 'weather_condition': 'Rain', 'wind_speed_mps': 1.63, 'date_time': datetime.datetime(2026, 7, 13, 9, 31, 58)}]


In [ ]:
df = pd.DataFrame(cleaned_data)
df

,city,temperature_c,humidity_pct,weather_condition,wind_speed_mps,date_time
0,Lagos,26.52,82,Rain,2.20,2026-07-13 09:28:08
1,Abuja,22.73,91,Clouds,2.53,2026-07-13 09:28:34
2,Ibadan,26.29,83,Rain,1.63,2026-07-13 09:31:58


In [ ]:
df.to_csv("weather_data.csv", index=False)

## Task 3: Load
Saving the cleaned dataset to CSV and SQLite for future analysis.

In [ ]:
import sqlite3

conn = sqlite3.connect("weather_data.db")
df.to_sql("weather", conn, if_exists="replace", index=False)
conn.close()

## Task 4: Analysis
Comparing temperature, humidity, and weather conditions across cities.

In [ ]:
hottest = df.loc[df["temperature_c"].idxmax()]
coldest = df.loc[df["temperature_c"].idxmin()]
most_humid = df.loc[df["humidity_pct"].idxmax()]

print("Hottest city:", hottest["city"], hottest["temperature_c"])
print("Coldest city:", coldest["city"], coldest["temperature_c"])
print("Most humid city:", most_humid["city"], most_humid["humidity_pct"])

Hottest city: Lagos 26.52
Coldest city: Abuja 22.73
Most humid city: Abuja 91
